In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import cv2
import json
import onnxruntime as ort
import seaborn as sns
from pathlib import Path
from itertools import chain
from tqdm.notebook import tqdm
from PIL import Image
import numpy as np
from sklearn.cluster import DBSCAN
import folium
import matplotlib.cm as cm
import matplotlib.colors as mcolors
from pyproj import Transformer
from collections import defaultdict

In [ ]:
result_json_dir = r'E:\data\202502_signboard\result\out-small'
image_dir = r'E:\data\202502_signboard\result\task0812\results\404'
result_json_vis_dir = r'E:\data\202502_signboard\result\out-small_vis'
reid_model_path = r'E:\repository\isds\cdu_seg_risk6\cdu_mount\reid_model.onnx'
os.makedirs(result_json_vis_dir, exist_ok=True)

cam_list = ['DA5148683', 'DA5324655', 'DA4930148', 'DA5324645', 'DA5148680', 'DA6102933']
eps = 10
min_samples = 1
reid_threshold=0.4
max_frames = 99999

In [ ]:
class FastReID_ONNX:
    def __init__(self, model_path):
        self.session = ort.InferenceSession(model_path, providers=["CUDAExecutionProvider"])
        self.input_name = self.session.get_inputs()[0].name

    def preprocess(self, img):
        img = cv2.resize(img, (128, 256), interpolation=cv2.INTER_CUBIC)
        img = img.astype("float32").transpose(2, 0, 1)[np.newaxis]
        return img

    def normalize(self, nparray, order=2, axis=-1):
        """Normalize a N-D numpy array along the specified axis."""
        norm = np.linalg.norm(nparray, ord=order, axis=axis, keepdims=True)
        return nparray / (norm + np.finfo(np.float32).eps)

    def extract(self, patch):
        input_tensor = self.preprocess(patch)
        feat = self.session.run(None, {self.session.get_inputs()[0].name: input_tensor})[0]
        feat = self.normalize(feat, axis=1)
        return feat


In [ ]:
class GlobalIDManager:
    def __init__(self, max_frames=30, reid_threshold=0.5, max_num_repeat=10, same_camera_match=True):
        self.track_buffer = []
        self.max_frames = max_frames
        self.reid_threshold = reid_threshold
        self.frame_id_map = {}
        self.global_id_map = {}
        self.global_id_counts = {}
        self.max_num_repeat = max_num_repeat
        self.next_global_id = 1
        self.same_camera_match = same_camera_match
        self.same_score_threshold = 0.95

    def _cosine_sim(self, a, b):
        return np.dot(a, b.T)

    def update(self, cam_id, timestamp, local_id, bbox, embedding):
        best_match = None
        best_score = -1
        frame_str = f'{timestamp}_{cam_id}'
        if frame_str in self.frame_id_map:
            self.frame_id = self.frame_id_map[frame_str]
        else:
            self.frame_id_map[frame_str] = len(self.frame_id_map)
            self.frame_id = self.frame_id_map[frame_str]

        current_image_key = f"{frame_str}_{local_id}"

        for track in self.track_buffer:
            # 是否匹配同一个摄像头
            if not self.same_camera_match and track['camera_id'] == cam_id:
                continue
            if abs(track['frame_id'] - self.frame_id) > self.max_frames:
                continue  # 超出缓存窗口
            if len(self.global_id_counts.get(track['global_id'], set())) >= self.max_num_repeat:
                continue  # 该全局ID已达到最大图像数量
            if embedding is not None and track['embedding'] is not None:
                sim = self._cosine_sim(embedding, track['embedding'])
                if sim > self.reid_threshold and sim > best_score:
                    best_match = track
                    best_score = sim
        if best_match:
            global_id = best_match['global_id']
            # 只有当这是一个新的图像时才增加计数
            if current_image_key not in self.global_id_counts.get(global_id, set()):
                self.global_id_counts[global_id] = self.global_id_counts.get(global_id, set())
                self.global_id_counts[global_id].add(current_image_key)
        else:
            global_id = self.next_global_id
            self.next_global_id += 1

            self.global_id_counts[global_id] = self.global_id_counts.get(global_id, set())
            self.global_id_counts[global_id].add(current_image_key)
            self.global_id_map[current_image_key] = global_id


        # 添加当前目标到缓存
        self.track_buffer.append({
            'frame_id': self.frame_id,
            'camera_id': cam_id,
            'local_id': local_id,
            'bbox': bbox,
            'embedding': embedding,
            'global_id': global_id
        })

        # 控制缓存大小（按帧）
        self._prune_buffer(self.frame_id)

        return global_id

    def _prune_buffer(self, current_frame_id):
        # 删除早于当前帧 max_frames 的条目
        self.track_buffer = [
            t for t in self.track_buffer
            if current_frame_id - t['frame_id'] <= self.max_frames
        ]

In [ ]:
reid_model = FastReID_ONNX(reid_model_path)

In [ ]:

def img_crop(input_path, output_path, bbox):
    img = Image.open(input_path)
    img = img.crop(bbox)
    if img.size[0] == 0 or img.size[1] == 0:
        return False
    else:
        img.save(output_path)
        return img

In [ ]:
def record_process(record, cam_record_id, timestamp, input_image_path, vis_path):
    if len(record['slam_center']) != 3:
        return None
    patch = img_crop(input_image_path, vis_path, record['box'])
    if not patch:
        return None
    record['timestamp'] = timestamp
    record['cam_record_id'] = cam_record_id
    record['vis_path'] = vis_path
    record['embedding'] = reid_model.extract(np.array(patch))
    return record

In [ ]:
result_json_list = os.listdir(result_json_dir)
result_datas = []
drop_count = 0
for result_json_file in tqdm(result_json_list):
    result_json_path = os.path.join(result_json_dir, result_json_file)
    timestamp = Path(result_json_file).stem
    with open(result_json_path, 'r', encoding='utf-8') as f:
        result_data = json.load(f)
        for cam_id, result_data_single_camera in enumerate(result_data):
            input_image_dir = os.path.join(image_dir, f'input_{cam_id+1}')
            input_image_path = os.path.join(input_image_dir, f'{cam_list[cam_id]}_{timestamp}.jpg')
            for record_id, record in enumerate(result_data_single_camera):
                cam_record_id = f'{cam_id}_{record_id}'
                vis_path = os.path.join(result_json_vis_dir, f'{timestamp}_{cam_record_id}.jpg')
                record = record_process(record, cam_record_id, timestamp, input_image_path, vis_path)
                if record is not None:
                    result_datas.append(record)
                else:
                    drop_count += 1
                    print(f'drop {drop_count}, timestamp: {timestamp}, cam_record_id: {cam_record_id}')
print(f'总记录数: {len(result_datas)}, drop: {drop_count}')

In [ ]:
# 在HK80上做聚类
slam_centers = np.array([d["slam_center"] for d in result_datas])[:, :2]

db = DBSCAN(eps=eps, min_samples=min_samples, n_jobs=-1).fit(slam_centers)
labels_cluster = db.labels_  # shape=(N,3)
cluster_centers = np.unique(labels_cluster[labels_cluster != -1], axis=0)

print(f"原始点数: {len(slam_centers)}, 聚类簇数: {len(cluster_centers)}")

In [ ]:
# 用reid做聚类
labels_reid = [None] * len(result_datas)

# 先按簇分组
cluster_groups = defaultdict(list)
for i, lbl in enumerate(labels_cluster):
    cluster_groups[lbl].append(i)

for lbl, idx_list in cluster_groups.items():
    # ✅ 每个簇创建一个新的 ReID 管理器
    reid_manager = GlobalIDManager(max_frames=max_frames, reid_threshold=reid_threshold)
    
    for idx in idx_list:
        d = result_datas[idx]
        global_id = reid_manager.update(
            cam_id=d.get("cam_id", 0),
            timestamp=d.get("timestamp", 0),
            local_id=d.get("local_id", idx),
            bbox=d.get("bbox", None),
            embedding=d.get("embedding", None),
        )
        # 用 (簇号, 簇内id) 组合生成唯一id（便于后续区分）
        labels_reid[idx] = f"{lbl}_{global_id}"  # 或者保留两列分别存
        
# 得到唯一 reid label
labels_reid_uniques = sorted(set(labels_reid))

# ✅ 建立 remap 映射
label_to_remap = {lbl: i for i, lbl in enumerate(labels_reid_uniques)}
labels_reid_remap = [label_to_remap[lbl] for lbl in labels_reid]

print(f"原始点数: {len(result_datas)}, reid簇数: {len(labels_reid_uniques)}")

In [ ]:
for i, d in enumerate(result_datas):
    d["cluster_label"] = int(labels_cluster[i])
    d["reid_label"] = labels_reid[i]            # 字符串，如 "3_800001"
    d["reid_label_remap"] = labels_reid_remap[i]  # 连续整数，如 0, 1, 2, ...

In [ ]:
colormap_cluster = cm.get_cmap("hsv", len(cluster_centers))
color_map_dict_cluster = {lbl: mcolors.to_hex(colormap_cluster(i)) for i, lbl in enumerate(cluster_centers)}
colormap_reid = cm.get_cmap("hsv", len(labels_reid_uniques))
color_map_dict_reid = {i: mcolors.to_hex(colormap_reid(i)) for i in range(len(labels_reid_uniques))}

In [ ]:
def single_vis(label_field, color_map_dict, result_datas=result_datas,  save_file="map1.html"):
    transformer = Transformer.from_crs("EPSG:2326", "EPSG:4326", always_xy=True)

    pre_slam_point = result_datas[0]['slam_center']
    lon, lat = transformer.transform(pre_slam_point[0], pre_slam_point[1])
    m = folium.Map(location=[lat, lon], zoom_start=18, max_zoom=22)

    for idx, d in enumerate(result_datas):
        label = d[label_field]
        color = color_map_dict[label]
        e, n, h = d["slam_center"]
        lon, lat = transformer.transform(e, n)

        img_path = d["vis_path"]
        img_uri = Path(img_path).resolve().as_uri()  # e.g. file:///C:/.../images/2025....jpg
        # img_uri = os.path.relpath(d["vis_path"], start=os.path.dirname(save_file))
        # popup_html = f'<img src="{img_uri}" width="180">'
        popup_html = f"""
        <div style="width:200px">
            <p><b>ID:</b> {label}</p>
            <p><b>Lat:</b> {e:.2f} m</p>
            <p><b>Lon:</b> {n:.2f} m</p>
            <p><b>Height:</b> {h:.2f} m</p>
            <img src="{img_uri}" width="180">
        </div>
        """

        folium.CircleMarker(
            [lat, lon],
            radius=8,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.8,
            popup=folium.Popup(popup_html, max_width=250)
        ).add_to(m)

    m.save(save_file)

    print(f"✅ get map: {save_file}")

In [ ]:
def point_vis(label_field, color_map_dict, result_datas=result_datas,  save_file="map2.html"):
    transformer = Transformer.from_crs("EPSG:2326", "EPSG:4326", always_xy=True)

    # 1. 按 label 分组
    cluster_groups = defaultdict(list)
    for idx, d in enumerate(result_datas):
        cluster_groups[d[label_field]].append(d)

    # 2. 初始化地图（以第一个簇中心为起点）
    first_label = next(iter(cluster_groups))
    first_center = np.array([g["slam_center"] for g in cluster_groups[first_label]]).mean(axis=0)
    lon, lat = transformer.transform(first_center[0], first_center[1])
    m = folium.Map(location=[lat, lon], zoom_start=18, max_zoom=22)

    # 3. 为每个簇绘制一个点
    for lbl, group in cluster_groups.items():
        color = color_map_dict[lbl]
        points = np.array([g["slam_center"] for g in group])
        center = points.mean(axis=0)
        lon, lat = transformer.transform(center[0], center[1])

        # 拼接所有图片 HTML
        img_tags = []
        for g in group:
            img_uri = Path(g["vis_path"]).resolve().as_uri()
            # img_uri = os.path.relpath(d["vis_path"], start=os.path.dirname(save_file))
            img_tags.append(f'<img src="{img_uri}" width="180">')
        images_html = "<br>".join(img_tags)

        popup_html = f"""
        <div style="width:200px">
            <p><b>Cluster ID:</b> {lbl}</p>
            <p><b>Points:</b> {len(group)}</p>
            {images_html}
        </div>
        """

        folium.CircleMarker(
            [lat, lon],
            radius=10,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.9,
            popup=folium.Popup(popup_html, max_width=300)
        ).add_to(m)

    m.save(save_file)
    print(f"✅ get map: {save_file}")

In [ ]:
single_vis(label_field='cluster_label', color_map_dict=color_map_dict_cluster, save_file='map11.html')
point_vis(label_field='cluster_label', color_map_dict=color_map_dict_cluster,  save_file='map12.html')
single_vis(label_field='reid_label_remap', color_map_dict=color_map_dict_reid, save_file='map21.html')
point_vis(label_field='reid_label_remap', color_map_dict=color_map_dict_reid, save_file='map22.html')